In [21]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.pytorch
import torch 
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

In [22]:
df = pd.read_csv('../data/data_preprocessed.csv')
df['returns'] = np.log(df['CLOSE'] / df['CLOSE'].shift(1))
df.dropna(inplace=True)
df

,date,views,number_of_news,sentiment_score,CLOSE,VOLUME,returns
1,2006-07-20,501,17,4,203.26,1141288.0,-0.003389
2,2006-07-21,1038,22,3,202.00,1864106.0,-0.006218
3,2006-07-24,553,17,8,201.99,794152.0,-0.000050
4,2006-07-25,756,27,7,202.69,1674834.0,0.003460
5,2006-07-26,1287,30,4,203.10,1009200.0,0.002021
...,...,...,...,...,...,...,...
4857,2025-12-15,168207,50,5,409.65,3688415.0,0.018229
4858,2025-12-16,70523,37,9,413.00,3507863.0,0.008144
4859,2025-12-17,129196,47,6,411.45,2995120.0,-0.003760
4860,2025-12-18,152050,50,2,409.25,5504714.0,-0.005361


In [23]:
class TimeSeriesDataset(Dataset):
    def __init__(self, data, seq_len, step, target_col, feature_col, scaler_features, scaler_target):
        self.seq_len = seq_len
        self.step = step

        self.features = data[feature_col].values.astype(np.float32)
        self.target = data[target_col].values.astype(np.float32).reshape(-1, 1)

        # scaler 
        self.scaler_features = scaler_features
        self.scaler_target = scaler_target

        self.features = self.scaler_features.transform(self.features)
        self.target = self.scaler_target.transform(self.target) 

        #self.feature_means = np.mean(self.features, axis=0)
        #self.feature_vars = np.var(self.features, axis=0)

    
    def __len__(self):
        return (len(self.features) - self.seq_len) // self.step
    
    
    def __getitem__(self, idx):
        start_idx = idx * self.step
        end_idx = start_idx + self.seq_len

        x = self.features[start_idx:end_idx]
        y = self.target[end_idx]

        return torch.from_numpy(x), torch.from_numpy(y)

In [24]:
# Параметры для настройки
#   -   #   -   #   -   #   -   #   -   #
train_size = int(len(df) * 0.8)
train_df = df.iloc[:train_size]
val_df = df.iloc[train_size:]

seq_len = 10
step = 1
target_col = ['returns']
feature_col = ['sentiment_score', 'VOLUME']
scaler_features = StandardScaler().fit(train_df[feature_col])
scaler_target = StandardScaler().fit(train_df[target_col])

batch_size = 32
#   -   #   -   #   -   #   -   #   -   #

train_ds = TimeSeriesDataset(data=train_df,
                             seq_len=seq_len,
                             step=step,
                             target_col=target_col,
                             feature_col=feature_col,
                             scaler_features=scaler_features,
                             scaler_target=scaler_target)

val_ds = TimeSeriesDataset(data=val_df,
                           seq_len=seq_len,
                           step=step,
                           target_col=target_col,
                           feature_col=feature_col,
                           scaler_features=scaler_features,
                           scaler_target=scaler_target)


train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

c:\Users\Ваня\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\Ваня\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\Ваня\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\Ваня\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [25]:
class GRUPredictor(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, output_dim, dropout=0.2):
        super(GRUPredictor, self).__init__()
        
        self.gru = nn.GRU(
            input_size=input_dim, 
            hidden_size=hidden_dim, 
            num_layers=num_layers, 
            batch_first=True, 
            dropout=dropout if num_layers > 1 else 0
        )
        
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # x: (batch_size, seq_len, input_dim)
        # out: (batch_size, seq_len, hidden_dim)
        out, _ = self.gru(x)
        
        out = self.fc(out[:, -1, :]) 
        return out

In [ ]:
def train(model, train_loader, val_loader, epochs, lr, weight_decay, params_to_log=None):
    if mlflow.active_run():
        mlflow.end_run()

    with mlflow.start_run() as run:
        mlflow.log_param("lr", lr)
        mlflow.log_param("epochs", epochs)
        mlflow.log_param("weight_decay", weight_decay)
        if params_to_log:
            mlflow.log_params(params_to_log)

        criterion = nn.MSELoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model.to(device)

        for epoch in range(epochs):
            model.train()
            train_loss = 0
            for x_batch, y_batch in train_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)

                optimizer.zero_grad()
                outputs = model(x_batch)
                loss = criterion(outputs, y_batch)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()

            model.eval()
            val_loss = 0
            with torch.no_grad():
                for x_val, y_val in val_loader:
                    x_val, y_val = x_val.to(device), y_val.to(device)
                    preds = model(x_val)
                    val_loss += criterion(preds, y_val).item()
            
            avg_train_loss = train_loss / len(train_loader)
            avg_val_loss = val_loss / len(val_loader)
            mlflow.log_metric("train_loss", avg_train_loss, step=epoch)
            mlflow.log_metric("val_loss", avg_val_loss, step=epoch)

            print(f'Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f}')

        mlflow.pytorch.log_model(model, name="model")

Эксперименты с моделью

In [ ]:
mlflow.set_experiment("Stock-Analysis-GRU")
model_params = {'hidden_dim': 32, 'num_layers': 2}
model = GRUPredictor(input_dim=len(feature_col), 
                     hidden_dim=model_params['hidden_dim'], 
                     num_layers=model_params['num_layers'], 
                     output_dim=1)

train(model=model, 
      train_loader=train_loader, 
      val_loader=val_loader, 
      epochs=10, 
      lr=1e-4,
      weight_decay=1e-5,
      params_to_log=model_params)